In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, udf, hour, broadcast, count
from pyspark.sql.types import StringType
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline


spark = SparkSession.builder \
    .appName("ChicagoCrimesAnalysis") \
    .config("spark.sql.shuffle.partitions", "10") \
    .getOrCreate()

df = spark.read.csv("chicago_crimes_sample.csv", header=True, inferSchema=True)

# Usuwanie duplikatów i braków w kluczowych kolumnach
df_cleaned = df.dropDuplicates() \
               .dropna(subset=["id", "date", "primary_type"])

df_cleaned = df_cleaned.withColumn(
    "parsed_date",
    to_timestamp(col("date"), "yyyy-MM-dd'T'HH:mm:ss.SSS")
)

df_cleaned = df_cleaned.filter(col("parsed_date").isNotNull())

# 3. UDF: Klasyfikacja pory dnia
def classify_time_of_day(h):
    if h is None: return "Unknown"
    if 6 <= h < 12: return "Morning"       # 06:00 - 11:59
    elif 12 <= h < 18: return "Afternoon"  # 12:00 - 17:59
    elif 18 <= h < 22: return "Evening"    # 18:00 - 21:59
    else: return "Night"                   # 22:00 - 05:59

# Rejestracja UDF
time_of_day_udf = udf(classify_time_of_day, StringType())

# Dodanie nowej kolumny
df_with_time = df_cleaned.withColumn("hour", hour(col("parsed_date"))) \
                         .withColumn("time_of_day", time_of_day_udf(col("hour")))


districts_data = [("008", "Southwest"), ("009", "Southwest"), ("012", "Near West"), ("014", "Northwest")]
districts_df = spark.createDataFrame(districts_data, ["district", "region"])

# Użycie broadcast()
df_optimized = df_with_time.join(broadcast(districts_df), "district", "left")

df_optimized.cache()
print(f"Liczba wierszy po czyszczeniu: {df_optimized.count()}")

# Zapis wyniku do Parquet z partycjonowaniem po roku
df_optimized.write \
    .partitionBy("year") \
    .mode("overwrite") \
    .parquet("chicago_crimes_parquet")

print("\n--- Analiza według typu przestępstwa ---")
crimes_by_type = df_optimized.groupBy("primary_type").count().orderBy(col("count").desc())
crimes_by_type.explain() # Wyświetla plan fizyczny i logiczny wykonania
crimes_by_type.show(5)

print("\n--- Analiza według lokalizacji ---")
crimes_by_location = df_optimized.groupBy("location_description").count().orderBy(col("count").desc())
crimes_by_location.explain()
crimes_by_location.show(5)

print("\n--- Analiza według czasu (pory dnia) ---")
crimes_by_time = df_optimized.groupBy("time_of_day").count().orderBy(col("count").desc())
crimes_by_time.explain()
crimes_by_time.show(5)

# Model MLlib
print("\n--- Budowa modelu klasyfikacyjnego MLlib ---")
top_crimes = [row['primary_type'] for row in crimes_by_type.limit(5).collect()]
df_ml = df_optimized.filter(col("primary_type").isin(top_crimes)) \
                    .dropna(subset=["location_description", "hour"])

# indeksowanie
location_indexer = StringIndexer(inputCol="location_description", outputCol="location_index", handleInvalid="keep")
label_indexer = StringIndexer(inputCol="primary_type", outputCol="label")

# sklejenie cech w jeden wektor
assembler = VectorAssembler(inputCols=["hour", "location_index"], outputCol="features")

# zdefiniowanie klasyfikatora
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=20, maxBins=150)

# ułożenie operacji w Pipeline
pipeline = Pipeline(stages=[location_indexer, label_indexer, assembler, rf])

# podział danych, trening i test
train_data, test_data = df_ml.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train_data)
predictions = model.transform(test_data)

#wyświetlenie próbki z predykcją
predictions.select("primary_type", "label", "location_description", "hour", "prediction", "probability").show(10)

Liczba wierszy po czyszczeniu: 50000

--- Analiza według typu przestępstwa ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [count#9204L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(count#9204L DESC NULLS LAST, 10), ENSURE_REQUIREMENTS, [plan_id=1372]
      +- HashAggregate(keys=[primary_type#7827], functions=[count(1)])
         +- Exchange hashpartitioning(primary_type#7827, 10), ENSURE_REQUIREMENTS, [plan_id=1369]
            +- HashAggregate(keys=[primary_type#7827], functions=[partial_count(1)])
               +- InMemoryTableScan [primary_type#7827]
                     +- InMemoryRelation [district#7833, id#7822, case_number#7823, date#7824, block#7825, iucr#7826, primary_type#7827, description#7828, location_description#7829, arrest#7830, domestic#7831, beat#7832, ward#7834, community_area#7835, fbi_code#7836, x_coordinate#7837, y_coordinate#7838, year#7839, updated_on#7840, latitude#7841, longitude#7842, location#7843, parsed_date#7845, hour#78